# 02 — Phân tích thăm dò dữ liệu (EDA) và kiểm định thống kê

**Nhóm 10 — IE313.F32.LT.CNTT.** Nguồn dữ liệu: Inside Airbnb Bangkok (`data/processed/bangkok_cleaned.csv`). Hạn nộp: 18/10/2026.

Ánh xạ bài giảng:
- **Bài 05 (EDA)**: Thống kê mô tả (Mean, Median, Std, IQR, Skewness, Kurtosis), Kiểm định tương quan Pearson ($r$, $p$-value), Bảng chéo 2D (Groupby, Pivot table), Đánh giá điều kiện và kiểm định One-way ANOVA ($F$, $p$-value).
- **Bài 02 (Tiền xử lý)**: Minh họa Feature Scaling (Simple feature scaling, Min-Max, Z-score), Binning (Equal-width vs Quantile), và One-Hot Encoding (Dummy variables) trên bản sao EDA cách ly hoàn toàn, không ghi đè baseline.
- **Ranh giới học thuật**: Tương quan không phải nhân quả. Các kiểm định thống kê mang tính khám phá (exploratory), lưu ý rủi ro gom cụm chủ nhà (host clustering).

Levene và qcut được tách nhãn **[MỞ RỘNG NGOÀI BÀI HỌC]**, xem `docs/EDA_METHODS.md` về nguồn, mục đích và giới hạn. EDA moments dùng bias=False; khác audit sơ bộ bias=True.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "workspaces/final-project"]
PROJECT = next(
    (
        p
        for p in candidates
        if (p / "data/source_manifest.json").is_file() and (p / "src/eda_stats.py").is_file()
    ),
    None,
)
if PROJECT is None:
    raise RuntimeError("Chạy từ gốc đồ án hoặc thư mục notebooks.")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.data_loader import compute_sha256  # noqa: E402
from src.eda_stats import (  # noqa: E402
    EXPECTED_BASELINE_SHA256,
    compute_anova_test,
    compute_neighbourhood_pivot,
    compute_pairwise_correlations,
    compute_price_distribution_stats,
    compute_room_type_stats,
    demonstrate_lesson02_wrangling,
    run_full_eda,
)

baseline_csv = PROJECT / "data/processed/bangkok_cleaned.csv"
actual_hash = compute_sha256(baseline_csv)
assert actual_hash == EXPECTED_BASELINE_SHA256, f"Hash baseline không khớp: {actual_hash}"
df = pd.read_csv(baseline_csv)

print(
    f"Xác thực baseline thành công: {df.shape[0]:,} dòng x {df.shape[1]} cột. SHA-256: {actual_hash}"
)

Xác thực baseline thành công: 28,987 dòng x 16 cột. SHA-256: 961a01ad225e35f989ec81f124925488e392baca2ef3800ff4028b4423d5f9a7


## 1. Khám phá biến mục tiêu trung tâm: Giá thuê theo đêm (RQ1)

Theo Bài 05 tr. 4–20, tính toán đầy đủ các đại lượng thống kê mô tả: độ tập trung (Mean, Median, Mode), độ phân tán (Min, Max, Range, IQR, Variance, Std, CV), và hình dạng phân phối (Skewness, Kurtosis).

In [2]:
st_price = compute_price_distribution_stats(df["price"])
df_rq1 = pd.DataFrame([st_price])
display(
    df_rq1[
        [
            "n",
            "min",
            "q1",
            "median",
            "mean",
            "q3",
            "max",
            "iqr",
            "std",
            "cv_pct",
            "skewness",
            "kurtosis",
            "outliers_count",
            "outliers_pct",
        ]
    ]
)

print(f"- Giá trung vị (Median): {st_price['median']:,.2f} THB/đêm")
print(
    f"- Giá trung bình (Mean): {st_price['mean']:,.2f} THB/đêm (cao gấp {st_price['mean'] / st_price['median']:.2f} lần Median; skewness = +{st_price['skewness']:.2f})"
)
print(f"- Khoảng biến thiên tứ phân vị (IQR): {st_price['iqr']:,.2f} THB/đêm")
print(
    f"- Số ngoại lai theo 1.5xIQR (> {st_price['upper_bound']:,.2f} THB): {st_price['outliers_count']:,} căn ({st_price['outliers_pct']}%)"
)

,n,min,q1,median,mean,q3,max,iqr,std,cv_pct,skewness,kurtosis,outliers_count,outliers_pct
0,28987,1.0,1041.47,1592.0,2955.96,2539.57,1320255.0,1498.1,19054.72,644.62,52.3111,3094.3914,2730,9.42


- Giá trung vị (Median): 1,592.00 THB/đêm
- Giá trung bình (Mean): 2,955.96 THB/đêm (cao gấp 1.86 lần Median; skewness = +52.31)
- Khoảng biến thiên tứ phân vị (IQR): 1,498.10 THB/đêm
- Số ngoại lai theo 1.5xIQR (> 4,786.72 THB): 2,730 căn (9.42%)


## 2. Giá theo loại phòng và ANOVA thăm dò (RQ2)

Mô tả nhóm bằng n, trung vị, Q1/Q3 và độ lệch. Skew không phải kiểm định chuẩn; không dùng p≥0,05 để chứng minh phương sai bằng nhau, không dùng ID duy nhất để chứng minh độc lập.

**[MỞ RỘNG NGOÀI BÀI HỌC]** Levene center=median chỉ chẩn đoán phương sai, không thay thế phương pháp chính trong Bài 05. Nguồn: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.levene.html ; host clustering giới hạn p-value. Xem `docs/EDA_METHODS.md`.

ANOVA tổng thể minh họa F/p thăm dò; không thực hiện so sánh cặp khi điều kiện suy luận chưa xác nhận.

In [3]:
rq2_stats = compute_room_type_stats(df)
df_room = pd.DataFrame.from_dict(rq2_stats["by_room_type"], orient="index")
display(df_room[["n", "group_n_pct", "q1", "median", "q3", "mean", "std", "skewness"]])
anova_res = compute_anova_test(df)
display(pd.DataFrame([anova_res["omnibus_test"]]))
print(anova_res["condition_checks"]["scientific_recommendation"])
print("Levene (mở rộng):", anova_res["condition_checks"]["homoscedasticity_evaluation"])
print("Host clustering trong baseline:", anova_res["condition_checks"]["independence_evaluation"])
print("So sánh cặp:", anova_res["pairwise_status"])

,n,group_n_pct,q1,median,q3,mean,std,skewness
Entire home/apt,19300,66.58,1141.18,1715.44,2796.00,3233.67,20539.19,48.1500
Hotel room,275,0.95,1163.50,1999.00,2893.60,3697.51,18110.71,16.1033
Private room,8914,30.75,925.36,1389.83,2165.12,2460.13,16014.59,65.8285
Shared room,498,1.72,332.90,431.22,592.50,659.08,1134.63,9.9755


,f_statistic,p_value,significant_at_alpha,alpha,caveat
0,5.932326,0.000486,True,0.05,Chỉ thăm dò; chưa xác nhận đủ điều kiện suy luận.


Chưa xác nhận mọi giả định; F/p chỉ minh họa thăm dò. Ưu tiên mô tả median/IQR; không suy rộng kiểm định hoặc kết luận nhân quả.
Levene (mở rộng): {'test': '[MỞ RỘNG NGOÀI BÀI HỌC] Levene (center=median)', 'statistic': 3.1229788630382243, 'p_value': 0.02478257830764562, 'equal_variance_assumption_satisfied': False, 'conclusion': 'Có bằng chứng chống giả thuyết phương sai bằng nhau, với giới hạn độc lập.'}
Host clustering trong baseline: {'unique_hosts': 7231, 'multi_listing_hosts': 3215, 'host_id_missing': 0, 'independence_confirmed': False, 'note': 'ID duy nhất không chứng minh độc lập; host clustering giới hạn diễn giải p-value.'}
So sánh cặp: SKIPPED_ASSUMPTIONS_NOT_CONFIRMED


## 3. Mối liên hệ tuyến tính giữa quy mô chỗ ở và giá thuê (RQ3)

Tính toán hệ số tương quan tuyến tính Pearson $r$ và giá trị $p$-value giữa 4 biến quy mô (`accommodates`, `bedrooms`, `bathrooms`, `beds`) và `price` theo Bài 05 tr. 24–27.
Báo cáo rõ số mẫu hợp lệ theo từng cặp (pairwise complete cases) và số dòng bị khuyết ở biến tương ứng.

In [4]:
corr_df = compute_pairwise_correlations(df)
display(corr_df)

print("NHẬN XÉT PHƯƠNG PHÁP LUẬN TƯƠNG QUAN (RQ3):")
for _, r in corr_df.iterrows():
    p_disp = "< 0.001" if r["p_value"] < 0.001 else f"= {r['p_value']:.4e}"
    print(
        f"- {r['feature']:15}: r = {r['pearson_r']:+.4f} (p {p_disp}, n hợp lệ = {r['n_valid']:,}, khuyết = {r['n_missing']:,}) -> {r['strength']}"
    )
print(
    "- p nhỏ không biểu thị tương quan mạnh; host clustering giới hạn suy luận. Chưa xác định nguyên nhân r thấp hay tác động của ngoại lệ."
)

,feature,target,n_total,n_valid,n_missing,missing_pct,pearson_r,p_value,strength,status
0,accommodates,price,28987,28987,0,0.00,0.1067,4.267508e-74,Yếu (Weak),COMPUTED
1,bedrooms,price,28987,25428,3559,12.28,0.0881,6.035805e-45,Yếu (Weak),COMPUTED
2,bathrooms,price,28987,26286,2701,9.32,0.0841,1.976676e-42,Yếu (Weak),COMPUTED
3,beds,price,28987,27717,1270,4.38,0.0688,1.901698e-30,Yếu (Weak),COMPUTED


NHẬN XÉT PHƯƠNG PHÁP LUẬN TƯƠNG QUAN (RQ3):
- accommodates   : r = +0.1067 (p < 0.001, n hợp lệ = 28,987, khuyết = 0) -> Yếu (Weak)
- bedrooms       : r = +0.0881 (p < 0.001, n hợp lệ = 25,428, khuyết = 3,559) -> Yếu (Weak)
- bathrooms      : r = +0.0841 (p < 0.001, n hợp lệ = 26,286, khuyết = 2,701) -> Yếu (Weak)
- beds           : r = +0.0688 (p < 0.001, n hợp lệ = 27,717, khuyết = 1,270) -> Yếu (Weak)
- p nhỏ không biểu thị tương quan mạnh; host clustering giới hạn suy luận. Chưa xác định nguyên nhân r thấp hay tác động của ngoại lệ.


## 4. Phân hóa địa lý theo quận hành chính và loại phòng (RQ4)

Xây dựng bảng chéo 2D (Pivot table) biểu diễn giá trung vị, số lượng mẫu $n$, và khoảng tứ phân vị IQR cho 50 quận hành chính Bangkok $\times$ 4 loại phòng theo Bài 05 tr. 28–36.
Cảnh báo các ô có số lượng mẫu rất nhỏ ($n < 5$) để tránh ngụy biện thống kê.

In [5]:
p_med, p_cnt, p_iqr, p_meta = compute_neighbourhood_pivot(df)
top_districts = df["neighbourhood_cleansed"].value_counts().head(15).index
print("Trung vị THB/đêm và số mẫu theo quận × loại phòng:")
display(p_med.loc[top_districts])
display(p_cnt.loc[top_districts])
print({key: p_meta[key] for key in ["total_pivot_cells", "empty_cells_count", "low_n_cells_count"]})
room = "Entire home/apt"
within_room = pd.DataFrame({"median": p_med[room], "n": p_cnt[room]})
within_room = within_room[within_room.n >= 20].sort_values("median")
print("So sánh trong cùng Entire home/apt; n≥20 là quy tắc hiển thị:")
display(pd.concat([within_room.head(5), within_room.tail(5)]))

Trung vị THB/đêm và số mẫu theo quận × loại phòng:


room_type,Entire home/apt,Hotel room,Private room,Shared room
neighbourhood_cleansed,,,,
Vadhana,2310.96,2122.00,1938.00,587.44
Huai Khwang,1711.80,11412.00,1437.74,315.00
Khlong Toei,1879.07,1940.00,1562.67,408.00
Ratchathewi,1806.46,1330.00,1733.00,492.99
Phra Nakhon,2918.50,1300.00,1358.00,332.90
Sathon,1496.25,2864.00,1200.00,420.00
Phra Khanong,1528.50,1942.34,1146.21,323.88
Bang Rak,2688.75,2140.50,1404.00,412.09
Suanluang,1035.10,NaN,1290.50,388.00


room_type,Entire home/apt,Hotel room,Private room,Shared room
neighbourhood_cleansed,,,,
Vadhana,3292,24,820,43
Huai Khwang,3245,13,478,21
Khlong Toei,2932,19,709,35
Ratchathewi,1089,31,527,43
Phra Nakhon,130,31,1023,135
Sathon,812,9,400,35
Phra Khanong,787,2,268,11
Bang Rak,616,44,365,28
Suanluang,599,0,226,6


{'total_pivot_cells': 200, 'empty_cells_count': 43, 'low_n_cells_count': 27}
So sánh trong cùng Entire home/apt; n≥20 là quy tắc hiển thị:


,median,n
neighbourhood_cleansed,,
Bang Phlat,678.03,284
Thon buri,940.14,396
Bang Na,969.48,586
Bang Sue,992.12,338
Lak Si,1022.00,67
Saphan Sung,2777.78,40
Parthum Wan,2874.83,440
Phra Nakhon,2918.50,130
Lat Phrao,3010.00,31


## 5. Minh họa kỹ thuật tiền xử lý Bài 02 (Scaling, Binning, Dummy)

Thực hiện minh họa các kỹ thuật của Bài 02 (TH01, Bài 02) trên một bản sao cách ly (`eda_sample`), không làm biến đổi tệp baseline và không rò rỉ dữ liệu vào mô hình hồi quy sau này:
1. Feature Scaling trên `accommodates`: Simple Feature Scaling, Min-Max Normalization, Z-score Standardization.
2. Binning trên `price`: So sánh phân khoảng đều (`pd.cut`) vs phân khoảng theo phân vị (`pd.qcut`).
3. Mã hóa biến giả (One-Hot Encoding) trên `room_type` qua `pd.get_dummies`.

**[MỞ RỘNG NGOÀI BÀI HỌC]** qcut minh họa phân khoảng theo phân vị; nguồn https://pandas.pydata.org/docs/reference/api/pandas.qcut.html . Nhóm trùng giá có thể không chia đủ khoảng. Không khẳng định ưu thế chung; xem `EDA_METHODS.md`.

In [6]:
l02_demo = demonstrate_lesson02_wrangling(df)
display(pd.DataFrame(l02_demo["scaling_summary"]))
print("So sánh 2 phương pháp Binning:")
print("- Equal-width binning (pd.cut):", l02_demo["binning_summary"]["equal_width_counts"])
print("- Quantile-based binning (pd.qcut):", l02_demo["binning_summary"]["quantile_counts"])
print(f"- Bài học sư phạm: {l02_demo['binning_summary']['pedagogical_lesson']}")

print("Mẫu dữ liệu One-Hot Encoding:")
display(pd.DataFrame(l02_demo["dummy_sample"]))

,original_accommodates,simple_feature_scaling,min_max_scaling,z_score_standardization
count,28987.0000,28987.0000,28987.0000,28987.0000
mean,3.0936,0.1934,0.1396,-0.0000
std,2.2403,0.1400,0.1493,1.0000
min,1.0000,0.0625,0.0000,-0.9345
25%,2.0000,0.1250,0.0667,-0.4882
50%,2.0000,0.1250,0.0667,-0.4882
75%,4.0000,0.2500,0.2000,0.4046
max,16.0000,1.0000,1.0000,5.7611


So sánh 2 phương pháp Binning:
- Equal-width binning (pd.cut): {'Low': 28980, 'High': 7, 'Medium': 0}
- Quantile-based binning (pd.qcut): {'Low': 9664, 'Medium': 9662, 'High': 9661}
- Bài học sư phạm: Khoảng đều mô tả chiều rộng giá, phân vị hướng tới số quan sát cân bằng; không có cách tốt hơn cho mọi mục tiêu. Xem số đếm thực tế; không dùng price bins làm predictor.
Mẫu dữ liệu One-Hot Encoding:


,room_type_Entire home/apt,room_type_Hotel room,room_type_Private room,room_type_Shared room
0,1,0,0,0
1,1,0,0,0
2,0,0,1,0
3,1,0,0,0
4,0,0,1,0


## 6. Xuất bản toàn bộ tệp thống kê máy đọc & Kiểm tra bàn giao

Thực thi hàm `run_full_eda()` để xuất bản toàn bộ 12 artifact thống kê sang `reports/stats/` có gắn metadata và hash xác thực.

In [7]:
meta_eda = run_full_eda(
    baseline_csv, PROJECT / "reports/stats", expected_sha256=EXPECTED_BASELINE_SHA256
)
assert (PROJECT / "reports/stats/rq1_price_distribution.json").is_file()
assert (PROJECT / "reports/stats/rq2_room_type_stats.csv").is_file()
assert (PROJECT / "reports/stats/rq3_size_correlation.csv").is_file()
assert (PROJECT / "reports/stats/rq4_neighbourhood_pivot_median.csv").is_file()
print("Toàn bộ artifact thống kê máy đọc đã sẵn sàng tại reports/stats/.")
print("Bước tiếp theo: notebook 03 — trực quan hóa biểu đồ 300 DPI theo 5 bước Bài 04.")

[*] 1/5 Đang tính toán thống kê phân phối giá (RQ1)...
[*] 2/5 Đang tính toán thống kê theo loại phòng & kiểm tra điều kiện ANOVA (RQ2)...
[*] 3/5 Đang tính toán tương quan Pearson các biến quy mô (RQ3)...
[*] 4/5 Đang xây dựng bảng chéo 2D theo quận và loại phòng (RQ4)...


[*] 5/5 Đang thực hiện minh họa kỹ thuật Bài 02 (Scaling, Binning, Dummy)...
[✓] Đã xuất bản thành công toàn bộ 12 tệp thống kê tại: /var/www/lienthongdaihoc/IE313.F32.LT.CNTT.PhanTichVaTrucQuanDuLieu/workspaces/final-project/reports/stats
Toàn bộ artifact thống kê máy đọc đã sẵn sàng tại reports/stats/.
Bước tiếp theo: notebook 03 — trực quan hóa biểu đồ 300 DPI theo 5 bước Bài 04.
